# Etapa 3 — Comparação entre rede neural pequena e profunda

**Inteligência Artificial · Ciência da Computação · PUC Minas · Trabalho Prático 3**

Comparamos dois perceptrons multicamadas (MLP) para identificar conexões anômalas. A rede pequena tem uma camada oculta com 16 neurônios. A rede profunda tem três camadas ocultas com 64, 32 e 16 neurônios.

Abra no Colab e escolha **Ambiente de execução → Executar tudo**. Forneça apenas o **treino.csv** oficial quando solicitado. Os dez ajustes de 120 épocas podem levar alguns minutos em CPU.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
import tempfile
from hashlib import sha256
from IPython.display import Markdown, display

candidatos = [
    Path("data/raw/treino.csv"),
    Path("treino.csv"),
    Path("/content/treino.csv"),
]
treino_csv = next((arquivo for arquivo in candidatos if arquivo.is_file()), None)
if treino_csv is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            "Coloque o treino.csv oficial na pasta inicial ou em data/raw e execute novamente."
        ) from erro
    treino_csv = Path("data/raw/treino.csv")
    treino_csv.parent.mkdir(parents=True, exist_ok=True)
    print("Selecione o treino.csv oficial do Canvas.")
    files.upload_file(str(treino_csv))

treino_csv = treino_csv.resolve()
with treino_csv.open(encoding="utf-8-sig", newline="") as arquivo:
    cabecalho = next(csv.reader(arquivo), [])
obrigatorias = {"class", "protocol_type", "service", "flag", "src_bytes", "total_bytes"}
if not obrigatorias.issubset(cabecalho):
    raise ValueError(
        f"O arquivo em {treino_csv} não tem o cabeçalho do treino oficial. "
        "Substitua-o pelo treino.csv correto e execute novamente."
    )
trabalho = Path(tempfile.mkdtemp(prefix="ia-etapa3-redes-"))
codigo_dados = trabalho / "dados_etapa2.py"
codigo_tecnicas = trabalho / "tecnicas_etapa2.py"
codigo_experimentos = trabalho / "avaliar_redes.py"
print("Base encontrada:", treino_csv.name)

In [ ]:
VERSOES = {
    "numpy": "2.5.4",
    "pandas": "3.0.1",
    "scipy": "1.18.1",
    "scikit-learn": "1.7.2",
    "imbalanced-learn": "0.14.1",
    "joblib": "1.6.0",
    "threadpoolctl": "3.8.0",
    "sklearn-compat": "0.1.6",
}

def consultar_versoes(ambiente):
    consulta = (
        "import importlib.metadata as m, json, sys; "
        "print(json.dumps({n: next((d.version for d in m.distributions(name=n)), None) "
        "for n in sys.argv[1:]}))"
    )
    resposta = subprocess.check_output(
        [sys.executable, "-c", consulta, *VERSOES], env=ambiente, text=True
    )
    return json.loads(resposta)

ambiente = os.environ.copy()
ambiente["PYTHONIOENCODING"] = "utf-8"
ambiente["PYTHONNOUSERSITE"] = "1"
ambiente["OPENBLAS_NUM_THREADS"] = "1"
ambiente["OMP_NUM_THREADS"] = "1"
if consultar_versoes(ambiente) != VERSOES:
    assinatura = sha256(json.dumps(VERSOES, sort_keys=True).encode()).hexdigest()[:12]
    bibliotecas = Path(tempfile.gettempdir()) / f"ia-etapa2-{sys.implementation.cache_tag}-{assinatura}"
    caminho_existente = ambiente.get("PYTHONPATH")
    ambiente["PYTHONPATH"] = os.pathsep.join(filter(None, (str(bibliotecas), caminho_existente)))
    if consultar_versoes(ambiente) != VERSOES:
        print("Instalando as versões dos experimentos. Aguarde a conclusão desta célula.", flush=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--only-binary=:all:", "--upgrade", "--target", str(bibliotecas),
             *[f"{nome}=={versao}" for nome, versao in VERSOES.items()]],
            check=True, env={**os.environ, "PYTHONNOUSERSITE": "1"},
        )
versoes_ativas = consultar_versoes(ambiente)
if versoes_ativas != VERSOES:
    raise RuntimeError(f"As versões do ambiente não coincidiram com as previstas: {versoes_ativas}")
print("Python:", sys.version.split()[0])
print("Bibliotecas:", versoes_ativas)

## Dados e preparação

Reutilizamos as divisões estratificadas por grupo das etapas anteriores. As duas redes usam a versão C, deduplicação apenas no ajuste, mediana nas ausências numéricas, categoria explícita nas ausências nominais, one-hot e padronização Z-score. Mantemos o oversampling da Etapa 2, de uma anomalia para dois registros normais. A amostragem é aplicada apenas aos exemplos usados para ajustar os pesos da rede. A validação conserva a proporção observada das classes.

As duas redes recebem a mesma preparação, taxa de aprendizado, tamanho de lote e número de épocas. A diferença pré-definida é o conjunto de camadas ocultas. Ao comparar com SVM ou Random Forest, os resultados correspondem aos pipelines de cada modelo, pois o oversampling foi aplicado às redes.

In [ ]:
%%writefile "{codigo_dados}"
"""Prepara os dados e cria divisoes agrupadas para as etapas 2 e 3."""

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

ALVO = "class"
COLUNAS_EXCLUIDAS_EXPERIMENTOS = (
    "num_outbound_cmds", "tempo_resposta_analista_min", "tipo_alerta_soc", "score_ids_legado",
)
SEMENTE = 42
NOMINAIS = ["protocol_type", "service", "flag", "protocolo_cod"]
COPIAS = ["src_bytes_kb", "dst_bytes_kb", "same_srv_pct", "protocolo_cod"]


def carregar_treino(caminho: str | Path) -> pd.DataFrame:
    """Lê o CSV oficial e valida a coluna alvo."""
    caminho = Path(caminho)
    if not caminho.is_file():
        raise FileNotFoundError(f"Base oficial não encontrada: {caminho}")
    dados = pd.read_csv(caminho)
    if ALVO not in dados or dados.empty:
        raise ValueError(f"A base deve ser não vazia e conter {ALVO!r}.")
    if dados[ALVO].isna().any() or not set(dados[ALVO].unique()).issubset({"normal", "anomaly"}):
        raise ValueError("A coluna class deve conter somente normal ou anomaly, sem ausências.")
    return dados.copy()


def identificar_taxas(dados: pd.DataFrame) -> list[str]:
    """Retorna apenas frações ``*_rate``; ``*_pct`` usa escala percentual."""
    return [c for c in dados.select_dtypes(include="number") if c.lower().endswith("rate")]


def preparar_base(dados):
    entradas = dados.drop(columns=["class", *COLUNAS_EXCLUIDAS_EXPERIMENTOS])
    for coluna in NOMINAIS:
        entradas[coluna] = entradas[coluna].astype("string").str.strip().str.casefold().replace("", pd.NA)

    invalidos = {}
    for coluna in ("duration", "src_bytes", "dst_bytes"):
        invalidos[coluna] = entradas[coluna].lt(0)
    for coluna in identificar_taxas(entradas):
        invalidos[coluna] = entradas[coluna].notna() & ~entradas[coluna].between(0, 1)
    invalidos["src_bytes_kb"] = invalidos["src_bytes"] | entradas["src_bytes_kb"].lt(0)
    invalidos["dst_bytes_kb"] = invalidos["dst_bytes"] | entradas["dst_bytes_kb"].lt(0)
    invalidos["total_bytes"] = invalidos["src_bytes"] | invalidos["dst_bytes"] | entradas["total_bytes"].lt(0)
    percentual = entradas["same_srv_pct"]
    invalidos["same_srv_pct"] = invalidos["same_srv_rate"] | (percentual.notna() & ~percentual.between(0, 100))
    for coluna, mascara in invalidos.items():
        entradas[coluna] = entradas[coluna].mask(mascara)

    representacoes = {
        "A": entradas,
        "B": entradas.drop(columns=COPIAS),
        "C": entradas.drop(columns=[*COPIAS, "total_bytes"]),
    }
    assert [m.shape[1] for m in representacoes.values()] == [45, 41, 40]
    grupos = representacoes["C"].groupby(list(representacoes["C"]), dropna=False, sort=False).ngroup()
    assert grupos.notna().all()
    return representacoes, grupos


def escolher_linhas(x, y, indices, regra):
    if regra == "todas":
        return indices
    chaves = x.iloc[indices].assign(rotulo=y[indices])
    mascara = np.ones(len(indices), dtype=bool)
    if regra in {"sem_conflitos", "sem_ambos"}:
        grupos = chaves.groupby(list(x), dropna=False, sort=False)["rotulo"]
        mascara &= grupos.transform("nunique").to_numpy() == 1
    if regra in {"sem_duplicatas", "sem_ambos"}:
        mascara &= ~chaves.duplicated().to_numpy()
    return indices[mascara]


def separar_divisoes(representacoes, y, grupos):
    divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
    particoes = list(divisor.split(representacoes["C"], y, grupos))
    reserva = particoes[-1][1]
    divisoes = []
    for fold, (treino, validacao) in enumerate(particoes, 1):
        if fold != 5:
            treino = np.setdiff1d(treino, reserva)
            assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[reserva])
            assert set(grupos.iloc[validacao]).isdisjoint(grupos.iloc[reserva])
        assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[validacao])
        divisoes.append((fold, treino, validacao))
    return divisoes


In [ ]:
%%writefile "{codigo_tecnicas}"
"""Transformacoes ajustadas apenas no treino de cada divisao."""

import numpy as np
import pandas as pd
from sklearn import config_context
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler
from dados_etapa2 import NOMINAIS, SEMENTE, escolher_linhas


def transformar_numericas(treino, validacao, metodo, escala):
    treino = treino.to_numpy(dtype=np.float64, na_value=np.nan).copy()
    validacao = validacao.to_numpy(dtype=np.float64, na_value=np.nan).copy()
    # Uma entrada sem observacoes no ajuste recebe zero em ambas as parcelas.
    # Valores observados apenas na validacao nao participam do ajuste do KNN.
    if metodo == "knn":
        vazias = np.isnan(treino).all(axis=0)
        treino[:, vazias] = 0.0
        validacao[:, vazias] = 0.0
    if metodo == "mediana_indicador":
        tem_ausencia = np.isnan(treino).any(axis=0)
        indicadores_treino = np.isnan(treino[:, tem_ausencia])
        indicadores_val = np.isnan(validacao[:, tem_ausencia])
    if metodo == "knn":
        # Limita o custo de vizinhança a uma amostra sem rótulos da parte de treino.
        normalizador = StandardScaler().fit(treino)
        amostra = np.random.default_rng(SEMENTE).choice(len(treino), size=min(2000, len(treino)), replace=False)
        imputador = KNNImputer(n_neighbors=5, keep_empty_features=True)
        imputador.fit(normalizador.transform(treino[amostra]))
        with config_context(working_memory=48):
            treino = normalizador.inverse_transform(imputador.transform(normalizador.transform(treino)))
            validacao = normalizador.inverse_transform(imputador.transform(normalizador.transform(validacao)))
    else:
        estrategia = "mean" if metodo == "media" else "median"
        imputador = SimpleImputer(strategy=estrategia, keep_empty_features=True)
        treino, validacao = imputador.fit_transform(treino), imputador.transform(validacao)
    normalizadores = {"zscore": StandardScaler, "robusta": RobustScaler, "minmax": MinMaxScaler}
    if escala in normalizadores:
        normalizador = normalizadores[escala]()
        treino, validacao = normalizador.fit_transform(treino), normalizador.transform(validacao)
    if metodo == "mediana_indicador":
        treino = np.column_stack((treino, indicadores_treino))
        validacao = np.column_stack((validacao, indicadores_val))
    return treino, validacao


def transformar_categoricas(treino, validacao, metodo, codificacao):
    estrategia = "most_frequent" if metodo == "moda" else "constant"
    colunas = list(treino)
    colunas_treino, colunas_validacao = [], []
    for coluna in colunas:
        valores_treino = treino[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        valores_validacao = validacao[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        # A moda nao existe quando toda a coluna do ajuste esta ausente.
        # Nesse caso a categoria explicita mantem a posicao da coluna.
        estrategia_coluna = "constant" if pd.isna(valores_treino).all() else estrategia
        imputador = SimpleImputer(strategy=estrategia_coluna, fill_value="__ausente__",
                                 keep_empty_features=True)
        colunas_treino.append(imputador.fit_transform(valores_treino))
        colunas_validacao.append(imputador.transform(valores_validacao))
    treino = np.column_stack(colunas_treino)
    validacao = np.column_stack(colunas_validacao)
    if codificacao == "inteiros":
        codificador = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        return codificador.fit_transform(treino), codificador.transform(validacao)
    cod_treino, cod_val = [], []
    for i, coluna in enumerate(colunas):
        agrupar = codificacao == "onehot_raros" and coluna == "service"
        codificador = OneHotEncoder(
            sparse_output=False,
            min_frequency=20 if agrupar else None,
            handle_unknown="infrequent_if_exist" if agrupar else "ignore",
        )
        cod_treino.append(codificador.fit_transform(treino[:, i:i + 1]))
        cod_val.append(codificador.transform(validacao[:, i:i + 1]))
    return np.column_stack(cod_treino), np.column_stack(cod_val)


def transformar(x, y, indices, validacao, config):
    indices = escolher_linhas(x, y, indices, config.linhas)
    treino, validacao = x.iloc[indices], x.iloc[validacao]
    categoricas = [c for c in NOMINAIS if c in x]
    numericas = x.columns.drop(categoricas).tolist()
    num_treino, num_val = transformar_numericas(treino[numericas], validacao[numericas], config.numericos, config.escala)
    cat_treino, cat_val = transformar_categoricas(treino[categoricas], validacao[categoricas], config.nominais, config.codificacao)
    return np.column_stack((num_treino, cat_treino)), np.column_stack((num_val, cat_val)), y[indices]


## Treinamento por retropropagação

Cada rede usa ReLU nas camadas ocultas, com lotes de 512 e a mesma semente. Em cada lote, a rede calcula a perda de entropia cruzada e usa **retropropagação** (backpropagation) para obter os gradientes dos pesos e dos vieses. O Adam usa esses gradientes para atualizar os parâmetros, com taxa de aprendizado inicial de 0,001. A rotina partial_fit mantém o estado do otimizador entre as épocas.

Cada arquitetura é treinada por exatamente **120 épocas em cada divisão**, sem parada antecipada. Registramos a perda de treino em todas as épocas e medimos o F1-macro da validação após a época 120. A quinta divisão já foi usada neste projeto como avaliação interna. As previsões do teste oficial ficam para a entrega final.

[Retropropagação no MLPClassifier](https://scikit-learn.org/1.7/modules/neural_networks_supervised.html#classification) · [Validação estratificada por grupos](https://scikit-learn.org/1.7/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)

In [ ]:
%%writefile "{codigo_experimentos}"
"""Compara duas arquiteturas MLP após exatamente 120 épocas por divisão."""

import json
from pathlib import Path
import sys
from types import SimpleNamespace

from imblearn.over_sampling import RandomOverSampler
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier
from threadpoolctl import threadpool_limits

from dados_etapa2 import SEMENTE, carregar_treino, preparar_base, separar_divisoes
from tecnicas_etapa2 import transformar

PREPARACAO = SimpleNamespace(
    linhas="sem_duplicatas", numericos="mediana", nominais="ausencia",
    codificacao="onehot", escala="zscore",
)
ARQUITETURAS = {"pequena": (16,), "profunda": (64, 32, 16)}
CLASSES = ["normal", "anomaly"]
ROTULOS = [0, 1]
EPOCAS = 120


def preparar_matrizes(x, y, treino, validacao):
    x_tr, x_val, y_tr = transformar(x, y, treino, validacao, PREPARACAO)
    x_tr, x_val = x_tr.astype(np.float32), x_val.astype(np.float32)
    if not (np.isfinite(x_tr).all() and np.isfinite(x_val).all()):
        raise ValueError("A transformação produziu valores não finitos.")
    amostrador = RandomOverSampler(sampling_strategy=0.5, random_state=SEMENTE)
    x_tr, y_tr = amostrador.fit_resample(x_tr, y_tr)
    return x_tr, x_val, y_tr, y[validacao]


def criar_rede(camadas):
    return MLPClassifier(
        hidden_layer_sizes=camadas, activation="relu", solver="adam", batch_size=512,
        alpha=0.0001, learning_rate_init=0.001, shuffle=True,
        # O estado avança a cada época, incluindo no embaralhamento dos lotes.
        random_state=np.random.RandomState(SEMENTE),
        early_stopping=False, max_iter=EPOCAS + 1, n_iter_no_change=EPOCAS + 1,
    )


def avaliar(camadas, matrizes, fold, nome):
    x_treino, x_validacao, y_treino, y_validacao = matrizes
    modelo = criar_rede(camadas)
    for epoca in range(1, EPOCAS + 1):
        # A retropropagação calcula os gradientes da perda em cada lote.
        # O Adam atualiza os parâmetros e mantém seu estado entre as épocas.
        modelo.partial_fit(x_treino, y_treino, classes=ROTULOS)
        if epoca % 30 == 0:
            print(f"Divisão {fold} · {nome} · época {epoca}/{EPOCAS} · perda {modelo.loss_:.4f}", flush=True)
    perdas = modelo.loss_curve_
    if len(perdas) != EPOCAS or modelo.t_ != EPOCAS * len(y_treino) or not np.isfinite(perdas).all():
        raise RuntimeError("O treinamento não completou as épocas com perdas finitas.")
    previsto = modelo.predict(x_validacao)
    relatorio = classification_report(
        y_validacao, previsto, labels=ROTULOS, target_names=CLASSES,
        output_dict=True, zero_division=0,
    )
    resultado = {
        "arquitetura": nome, "fold": fold, "epocas": len(perdas),
        "linhas_ajuste": len(y_treino), "linhas_validacao": len(y_validacao),
        "colunas": x_treino.shape[1],
        "parametros": sum(pesos.size + bias.size for pesos, bias in zip(modelo.coefs_, modelo.intercepts_)),
        "f1_macro": relatorio["macro avg"]["f1-score"],
        "recall_anomaly": relatorio["anomaly"]["recall"],
        "precisao_anomaly": relatorio["anomaly"]["precision"],
        "matriz": confusion_matrix(y_validacao, previsto, labels=ROTULOS).tolist(),
        "relatorio": relatorio,
    }
    historico = [
        {"fold": fold, "arquitetura": nome, "epoca": epoca, "perda_treino": float(perda)}
        for epoca, perda in enumerate(perdas, start=1)
    ]
    return resultado, historico


def executar(arquivo, pasta):
    dados = carregar_treino(arquivo)
    y = (dados["class"] == "anomaly").to_numpy(dtype=np.int32)
    representacoes, grupos = preparar_base(dados)
    divisoes = separar_divisoes(representacoes, dados["class"].to_numpy(), grupos)
    x = representacoes["C"]
    rodadas, reserva, historico = [], [], []
    for fold, treino, validacao in divisoes:
        matrizes = preparar_matrizes(x, y, treino, validacao)
        resultados_divisao = reserva if fold == 5 else rodadas
        for nome, camadas in ARQUITETURAS.items():
            resultado, curvas = avaliar(camadas, matrizes, fold, nome)
            resultados_divisao.append(resultado)
            historico.extend(curvas)
            print(f"Divisão {fold} · {nome} · F1-macro após 120 épocas {resultado['f1_macro']:.4f}", flush=True)

    tabela = pd.DataFrame(rodadas).drop(columns=["matriz", "relatorio"])
    medias = tabela.groupby("arquitetura", sort=False).agg(
        f1_macro=("f1_macro", "mean"), desvio_f1=("f1_macro", "std"),
        recall_anomaly=("recall_anomaly", "mean"), precisao_anomaly=("precisao_anomaly", "mean"),
    ).reset_index().sort_values(["f1_macro", "recall_anomaly"], ascending=False, kind="stable")
    tabela.to_csv(pasta / "redes_por_divisao.csv", index=False)
    pd.DataFrame(historico).to_csv(pasta / "redes_curvas_treino.csv", index=False)
    resultado = {
        "arquiteturas": ARQUITETURAS, "classes": CLASSES, "epocas": EPOCAS,
        "melhor": str(medias.iloc[0]["arquitetura"]), "medias": medias.to_dict(orient="records"),
        "reserva": reserva, "rodadas": tabela.to_dict(orient="records"),
        "divisoes": [
            {"fold": fold, "treino": len(treino), "validacao": len(val),
             "anomalias": int(y[val].sum())}
            for fold, treino, val in divisoes
        ],
    }
    (pasta / "redes_resultados.json").write_text(json.dumps(resultado, ensure_ascii=False), encoding="utf-8")


if __name__ == "__main__":
    with threadpool_limits(limits=1):
        executar(*map(Path, sys.argv[1:3]))

In [ ]:
subprocess.run(
    [sys.executable, "-u", str(codigo_experimentos), str(treino_csv), str(trabalho)],
    check=True, env=ambiente,
)
redes = json.loads((trabalho / "redes_resultados.json").read_text(encoding="utf-8"))

## Resultados

As duas primeiras tabelas mostram a média das quatro rodadas e o resultado de cada rodada após 120 épocas. A quinta divisão tem resultados e matrizes de confusão separados.

In [ ]:
def tabela(colunas, linhas):
    linhas = [colunas, ["---"] * len(colunas), *linhas]
    display(Markdown("\n".join("| " + " | ".join(map(str, linha)) + " |" for linha in linhas)))



nomes = {"pequena": "MLP pequena", "profunda": "MLP profunda"}
tabela(
    ("Arquitetura", "F1-macro médio ± desvio", "Recall de anomaly", "Precisão de anomaly"),
    [(nomes[r["arquitetura"]], f"{r['f1_macro']:.4f} ± {r['desvio_f1']:.4f}",
      f"{r['recall_anomaly']:.2%}", f"{r['precisao_anomaly']:.2%}") for r in redes["medias"]],
)
tabela(
    ("Divisão", "Arquitetura", "Épocas", "F1-macro na validação"),
    [(r["fold"], nomes[r["arquitetura"]], r["epocas"], f"{r['f1_macro']:.4f}")
     for r in redes["rodadas"]],
)
print("Maior média nas quatro rodadas:", nomes[redes["melhor"]])
tabela(
    ("Arquitetura na quinta divisão", "Épocas", "Parâmetros", "F1-macro", "Recall de anomaly", "Precisão de anomaly"),
    [(nomes[r["arquitetura"]], r["epocas"], r["parametros"], f"{r['f1_macro']:.4f}",
      f"{r['recall_anomaly']:.2%}", f"{r['precisao_anomaly']:.2%}") for r in redes["reserva"]],
)
for r in redes["reserva"]:
    display(Markdown(f"### {nomes[r['arquitetura']]} na quinta divisão"))
    tabela(
        ("Classe real", *(f"Previsto {classe}" for classe in redes["classes"])),
        [(classe, *linha) for classe, linha in zip(redes["classes"], r["matriz"])],
    )
print("CSV por divisão:", trabalho / "redes_por_divisao.csv")
print("CSV com o histórico do treinamento:", trabalho / "redes_curvas_treino.csv")

**Material da disciplina.** Rede Neural – Backpropagation e o notebook RNA, usado como referência de rede neural para classificação. As configurações desta comparação foram definidas para a base da competição.

**Apoio de IA.** Codex auxiliou na montagem e validação do experimento.